In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - Fungsi Qiskit oleh Kipu Quantum

> **Note:** * Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.

## Gambaran Umum
Dengan Iskay Quantum Optimizer dari Kipu Quantum, kamu bisa menangani masalah optimasi yang kompleks menggunakan komputer kuantum IBM&reg;. Solver ini memanfaatkan [algoritma bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) mutakhir dari Kipu yang hanya membutuhkan fungsi objektif sebagai input untuk secara otomatis menghasilkan solusi masalah. Optimizer ini bisa menangani masalah optimasi hingga 156 qubit, sehingga memungkinkan penggunaan semua qubit pada perangkat kuantum IBM. Optimizer menggunakan pemetaan 1-ke-1 antara variabel klasik dan qubit, yang memungkinkan kamu menangani masalah optimasi dengan hingga 156 variabel biner.

Optimizer ini memungkinkan penyelesaian masalah optimasi biner tanpa kendala. Selain formulasi QUBO (Quadratic Unconstrained Binary Optimization) yang umum digunakan, ia juga mendukung masalah optimasi orde tinggi (HUBO). Solver ini menggunakan algoritma kuantum non-variasional, yang menjalankan sebagian besar komputasi pada perangkat kuantum.

Berikut ini informasi lebih lanjut tentang algoritma yang digunakan dan panduan singkat cara menggunakan fungsi ini, beserta hasil benchmark pada berbagai instance masalah dengan ukuran dan kompleksitas yang berbeda-beda.

## Deskripsi
Optimizer ini adalah implementasi siap pakai dari algoritma optimasi kuantum mutakhir. Ia menyelesaikan masalah optimasi dengan menjalankan sirkuit kuantum yang sangat dikompresi pada perangkat keras kuantum. Kompresi ini dicapai dengan memasukkan suku-suku counterdiabatic ke dalam evolusi waktu yang mendasari sistem kuantum. Algoritma ini menjalankan beberapa iterasi eksekusi perangkat keras untuk mendapatkan solusi akhir dan menggabungkannya dengan pasca-pemrosesan. Langkah-langkah ini terintegrasi secara mulus ke dalam alur kerja Optimizer dan dieksekusi secara otomatis.

### Bagaimana cara kerja Quantum Optimizer?
Bagian ini menguraikan dasar-dasar algoritma bf-DCQO yang diimplementasikan. Pengenalan algoritma ini juga bisa ditemukan di [saluran YouTube Qiskit.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

Algoritma ini didasarkan pada evolusi waktu sistem kuantum yang berubah seiring waktu, di mana solusi masalah dikodekan dalam ground state sistem kuantum di akhir evolusi. Menurut [teorema adiabatik](https://en.wikipedia.org/wiki/Adiabatic_theorem), evolusi ini harus berlangsung lambat untuk memastikan sistem tetap berada di ground state-nya. Mendigitalisasi evolusi ini adalah dasar dari komputasi adiabatik kuantum terdigitalisasi (DQA) dan algoritma QAOA yang terkenal. Namun, evolusi lambat yang diperlukan tidak layak untuk ukuran masalah yang semakin besar karena menghasilkan kedalaman sirkuit yang semakin bertambah. Dengan menggunakan protokol counterdiabatic, kamu bisa menekan eksitasi yang tidak diinginkan yang terjadi selama waktu evolusi singkat sambil tetap berada di ground state. Di sini, mendigitalisasi waktu evolusi yang lebih singkat ini menghasilkan sirkuit kuantum dengan kedalaman lebih pendek dan lebih sedikit gate yang berpilin.

Sirkuit algoritma bf-DCQO biasanya menggunakan hingga sepuluh kali lebih sedikit gate yang berpilin dibandingkan DQA, dan tiga hingga empat kali lebih sedikit gate yang berpilin dibandingkan implementasi QAOA standar. Karena jumlah gate yang lebih sedikit, lebih sedikit kesalahan yang terjadi selama eksekusi sirkuit pada perangkat keras. Oleh karena itu, optimizer ini tidak memerlukan teknik seperti penekanan kesalahan atau mitigasi kesalahan. Mengimplementasikannya di versi mendatang dapat meningkatkan kualitas solusi lebih jauh.

Meskipun algoritma bf-DCQO menggunakan iterasi, ia bersifat non-variasional. Setelah setiap iterasi algoritma, distribusi state diukur. Distribusi yang diperoleh digunakan untuk menghitung apa yang disebut bias-field. Bias-field memungkinkan memulai iterasi berikutnya dari state energi mendekati solusi yang ditemukan sebelumnya. Dengan cara ini, algoritma bergerak dengan setiap iterasi menuju solusi berenergi lebih rendah. Biasanya, sekitar sepuluh iterasi sudah cukup untuk mencapai konvergensi ke solusi, secara keseluruhan memerlukan jumlah iterasi yang jauh lebih sedikit dibandingkan algoritma variasional, yang berada dalam urutan sekitar 100 iterasi.

Optimizer ini menggabungkan algoritma bf-DCQO dengan pasca-pemrosesan klasik. Setelah mengukur distribusi state, dilakukan pencarian lokal. Selama pencarian lokal, bit-bit dari solusi yang diukur dibalik secara acak. Setelah pembalikan, energi dari bitstring baru dievaluasi. Jika energinya lebih rendah, bitstring tersebut disimpan sebagai solusi baru. Pencarian lokal hanya berskala linier dengan jumlah qubit; karenanya, ini murah secara komputasi. Karena pasca-pemrosesan mengoreksi bitflip lokal, ia mengkompensasi kesalahan bit-flip yang sering merupakan hasil dari ketidaksempurnaan perangkat keras dan kesalahan pembacaan.

### Alur kerja
Berikut adalah skema alur kerja Quantum Optimizer.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Alur kerja Quantum Optimizer")

Dengan menggunakan Quantum Optimizer, penyelesaian masalah optimasi pada perangkat keras kuantum dapat disederhanakan menjadi:
* Merumuskan fungsi objektif dari masalah
* Mengakses Optimizer melalui Qiskit Functions
* Menjalankan Optimizer dan mengumpulkan hasilnya

## Benchmark
Metrik benchmark di bawah ini menunjukkan bahwa Optimizer secara efektif menangani masalah yang melibatkan hingga 156 qubit dan memberikan gambaran umum tentang akurasi dan skalabilitas optimizer di berbagai jenis masalah. Perhatikan bahwa metrik performa aktual mungkin bervariasi tergantung pada karakteristik masalah tertentu, seperti jumlah variabel, kepadatan dan lokalitas suku-suku dalam fungsi objektif, dan orde polinomial.

Tabel berikut mencantumkan rasio aproksimasi (AR), metrik yang didefinisikan sebagai berikut:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
di mana $C$ adalah fungsi objektif, $C_{\textrm{min}}$, $C_{\textrm{max}}$ adalah nilai minimum dan maksimumnya, dan $C^{*}$ adalah biaya solusi terbaik yang ditemukan, masing-masing. Oleh karena itu, AR=100\% berarti ground state dari masalah telah diperoleh.

| Contoh            | Jumlah qubit | Rasio Aproksimasi | Total waktu (s) | Penggunaan runtime (s) | Total jumlah shot | Jumlah iterasi |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Instance MaxCut dengan 28, 30, dan 32 qubit dijalankan pada ibm_sherbrooke. Instance dengan 80, 100, dan 120 dijalankan pada prosesor Heron r2.
- Instance HUBO juga dijalankan pada prosesor Heron r2.

Semua instance benchmark tersedia di GitHub (lihat [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Contoh untuk menjalankan instance ini bisa ditemukan di [Contoh 3: Instance benchmark](#example-3-benchmark-instances).

## Input dan output
### Input
Lihat tabel berikut untuk semua parameter input yang diterima oleh Quantum Optimizer. Bagian **Options** selanjutnya membahas lebih detail tentang `options` yang tersedia.

| Nama         | Tipe               | Deskripsi                                                                                                                                                                            | Wajib | Default                                                                                       | Contoh                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Koefisien dari masalah optimasi yang dirumuskan sebagai format QUBO/HUBO atau spin. Untuk informasi lebih lanjut tentang spesifikasi masalah, lihat [Format masalah yang diterima](#accepted-problem-formats)                                       | Ya      | T/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Tentukan apakah koefisien masalah dalam format biner (QUBO/HUBO) atau spin. Dua kemungkinannya adalah `"spin"` atau `"binary"`                                                   | Ya      | T/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Nama Backend untuk membuat kueri                                                                                                                                                  | Ya       | T/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Opsi untuk menangani pengiriman ke perangkat keras, seperti jumlah shot. Untuk detail lebih lanjut tentang konfigurasi opsi, lihat bagian [Options](#options) | Tidak       | Untuk melihat nilai default konfigurasi opsi lihat bagian [Options](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |

#### Format masalah yang diterima
Argumen `problem` dan `problem_type` mengkodekan masalah optimasi dalam bentuk

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
di mana

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Dengan memilih `problem_type = "binary"`, kamu menentukan bahwa fungsi biaya dalam format `binary`, yang berarti $D = {0,  1}^{n}$, yaitu fungsi biaya ditulis dalam formulasi QUBO/HUBO.
- Di sisi lain, dengan memilih `problem_type = "spin"`, fungsi biaya ditulis dalam formulasi Ising, di mana $D = {-1, 1}^{n}$.

Koefisien masalah harus dikodekan dalam dictionary sebagai berikut:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Perhatikan bahwa kunci dictionary harus berupa string yang berisi tuple integer tidak berulang yang valid.

#### Options
Iskay menyediakan kemampuan fine-tuning melalui parameter opsional. Meskipun nilai default bekerja dengan baik untuk sebagian besar masalah, kamu bisa menyesuaikan perilakunya untuk kebutuhan tertentu:

| Parameter | Tipe | Default | Deskripsi |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Pengukuran kuantum per iterasi (lebih tinggi = lebih akurat) |
| **num_iterations** | `int` | 10 | Iterasi algoritma (lebih banyak iterasi dapat meningkatkan kualitas solusi) |
| **use_session** | `bool` | True | Gunakan Session IBM untuk waktu antrian yang lebih singkat |
| **seed_transpiler** | `int` | None | Tetapkan untuk kompilasi sirkuit kuantum yang dapat direproduksi |
| **direct_qubit_mapping** | `bool` | False | Petakan qubit virtual langsung ke qubit fisik |
| **job_tags** | `List[str]` | None | Tag kustom untuk pelacakan job |
| **preprocessing_level** | `int` | 0 | Intensitas prapemrosesan masalah (0-3) - lihat detail di bawah |
| **postprocessing_level** | `int` | 2 | Level penyempurnaan solusi (0-2) - lihat detail di bawah |
| **transpilation_level** | `int` | 0 | Percobaan optimasi Transpiler (0-5) - lihat detail di bawah |
| **transpile_only** | `bool` | False | Analisis optimasi sirkuit tanpa menjalankan eksekusi penuh |

**Level Prapemrosesan (0-3)**: Sangat penting untuk masalah besar yang saat ini tidak dapat muat dalam waktu koherensi perangkat keras. Level prapemrosesan yang lebih tinggi menghasilkan kedalaman sirkuit yang lebih dangkal melalui aproksimasi dalam transpilasi masalah:
- **Level 0**: Tepat, sirkuit lebih panjang
- **Level 1**: Keseimbangan yang baik antara akurasi dan aproksimasi, hanya memotong gate dengan sudut di persentil terendah ke-10
- **Level 2**: Aproksimasi sedikit lebih tinggi, memotong gate dengan sudut di persentil terendah ke-20 dan menggunakan `approximation_degree=0.95` dalam transpilasi
- **Level 3**: Level aproksimasi maksimum, memotong gate di persentil terendah ke-30 dan menggunakan `approximation_degree=0.90` dalam transpilasi

**Level Transpilasi (0-5)**: Kendalikan percobaan optimasi transpiler lanjutan untuk kompilasi sirkuit kuantum. Ini dapat menyebabkan peningkatan overhead klasik, dan dalam beberapa kasus mungkin tidak mengubah kedalaman sirkuit. Nilai default `2` secara umum menghasilkan sirkuit terkecil dan relatif cepat.
- **Level 0**: Optimasi sirkuit DCQO yang terdekomposisi (layout, routing, scheduling)
- **Level 1**: Optimasi `PauliEvolutionGate` kemudian sirkuit DCQO yang terdekomposisi (max_trials=10)
- **Level 2**: Optimasi `PauliEvolutionGate` kemudian sirkuit DCQO yang terdekomposisi (max_trials=15)
- **Level 3**: Optimasi `PauliEvolutionGate` kemudian sirkuit DCQO yang terdekomposisi (max_trials=20)
- **Level 4**: Optimasi `PauliEvolutionGate` kemudian sirkuit DCQO yang terdekomposisi (max_trials=25)
- **Level 5**: Optimasi `PauliEvolutionGate` kemudian sirkuit DCQO yang terdekomposisi (max_trials=50)

**Level Pascapemrosesan (0-2)**: Kendalikan seberapa banyak optimasi klasik, mengkompensasi kesalahan bit-flip dengan jumlah pass greedy pencarian lokal yang berbeda:
- **Level 0**: 1 pass
- **Level 1**: 2 pass
- **Level 2**: 3 pass

**Mode transpile-only**: Kini tersedia untuk pengguna yang ingin menganalisis optimasi sirkuit tanpa menjalankan eksekusi algoritma kuantum penuh.

**Contoh konfigurasi kustom**: Berikut cara kamu mengonfigurasi Iskay dengan pengaturan berbeda:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Optimasi seed**: Perhatikan bahwa `seed_transpiler` diatur ke `None` secara default. Ini mengaktifkan proses optimasi otomatis transpiler. Ketika `None`, sistem akan memulai percobaan dengan beberapa seed dan memilih yang menghasilkan kedalaman sirkuit terbaik, memanfaatkan sepenuhnya parameter `max_trials` untuk setiap level transpilasi.

**Performa level transpilasi**: Meningkatkan jumlah `max_trials` dengan nilai `transpilation_level` yang lebih tinggi pasti akan meningkatkan waktu transpilasi, tetapi mungkin tidak selalu mengubah sirkuit akhir — ini sangat tergantung pada struktur dan kompleksitas sirkuit tertentu. Untuk beberapa sirkuit/masalah, bagaimanapun, perbedaan antara 10 percobaan (level 1) dan 50 percobaan (level 5) bisa sangat dramatis, jadi mengeksplorasi parameter ini mungkin menjadi kunci untuk berhasil menemukan solusi.

### Output
| Nama   | Tipe               | Deskripsi                                                           | Contoh                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Solusi dan metadata. Strukturnya bervariasi berdasarkan opsi `transpile_only`. |   Lihat "Isi dictionary hasil" di bawah      |

### Isi dictionary hasil
Struktur dictionary hasil tergantung pada mode eksekusi:

| Field         | Tipe               | Mode | Deskripsi                                                                                                             | Contoh |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Standard** | Solusi terpetakan yang diurutkan di mana kunci adalah indeks variabel (sebagai string) yang diurutkan secara numerik dan nilainya adalah nilai variabel yang sesuai (1/-1 untuk masalah spin, 1/0 untuk masalah biner).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Standard** | Informasi terperinci tentang solusi (lihat detail di bawah)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Standard** | Jenis masalah optimasi ('spin' atau 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Transpile-only** | Analisis sirkuit dan detail transpilasi (lihat detail di bawah) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Eksekusi standar
Ketika parameter opsional `transpile_only=False`:

**Dictionary solution_info:**
- **"bitstring"** (`str`): Representasi bitstring mentah dari solusi.
- **"cost"** (`float`): Nilai biaya/energi yang terkait dengan solusi.
- **"seed_transpiler"** (`int`): Seed acak yang digunakan untuk transpiler yang menghasilkan hasil ini.
- **"mapping"** (`Dict[int, int]`): Pemetaan qubit-ke-variabel asli yang digunakan dalam komputasi.
- **"qpu_time"** (`float`, opsional): Waktu eksekusi QPU dalam detik.

**Catatan pemetaan variabel:**
- Dictionary `solution` diperoleh dari bitstring solusi, menggunakan objek `mapping` untuk mengindeks variabel.
- Ketika `problem_type=spin` kita menggunakan penugasan $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Kunci dalam dictionary solusi adalah indeks variabel yang diurutkan secara numerik sebagai string.

#### Analisis transpilasi
Ketika parameter opsional `transpile_only=True`:

**Dictionary transpilation_info:**
- **"best_seed"** (`int`): Seed optimal yang ditemukan untuk transpilasi
- **"transpilation_time_seconds"** (`float`): Waktu yang dibutuhkan untuk proses transpilasi
- **"transpiled_circuit"** (`Dict`): Analisis sirkuit yang berisi:
  - **"depth"** (`int`): Kedalaman sirkuit (jumlah layer)
  - **"gate_count"** (`int`): Total jumlah gate dalam sirkuit
  - **"num_qubits"** (`int`): Jumlah qubit yang digunakan
  - **"width"** (`int`): Lebar sirkuit
  - **"operations"** (`Dict[str, int]`): Hitungan setiap jenis gate yang digunakan

**Penggunaan mode transpile-only:**
- Tersedia untuk pengguna yang ingin menganalisis optimasi sirkuit tanpa menjalankan eksekusi algoritma kuantum penuh.
- Berguna untuk analisis sirkuit, studi optimasi kedalaman, dan memahami efek transpilasi sebelum berkomitmen pada eksekusi penuh.

## Mulai
Dalam dokumentasi ini, kita akan melalui langkah-langkah menggunakan Iskay Quantum Optimizer. Dalam prosesnya kita akan dengan cepat menunjukkan cara memuat fungsi dari catalog dan cara mengonversi masalahmu ke input yang valid, sekaligus menunjukkan cara bereksperimen dengan berbagai parameter opsional yang berbeda.

Untuk contoh yang lebih detail, silakan lihat tutorial [Selesaikan masalah Market Split dengan Iskay Quantum Optimizer dari Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), di mana kita bekerja melalui seluruh proses menggunakan Iskay Solver untuk menangani masalah Market Split, yang mewakili tantangan alokasi sumber daya dunia nyata di mana pasar harus dipartisi ke dalam wilayah penjualan yang seimbang untuk memenuhi target permintaan yang tepat.

Autentikasi menggunakan API key-mu, yang bisa ditemukan di [dashboard IBM Quantum Platform](http://quantum.cloud.ibm.com/), dan pilih Qiskit Function sebagai berikut:

In [ ]:
# ruff: noqa: F821

> **Note:** Kode berikut mengasumsikan bahwa kamu telah menyimpan kredensialmu. Jika belum, ikuti instruksi di [simpan akun IBM Cloud-mu](/guides/functions#install-qiskit-functions-catalog-client) untuk mengautentikasi dengan API key-mu.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Contoh 1: Fungsi biaya sederhana

Perhatikan fungsi biaya dalam formulasi spin:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

di mana $(x_0, ..., x_4) \in {-1, 1}^5$.

Solusi untuk fungsi biaya sederhana ini adalah
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
dengan nilai minimum $C^{*} = -6$

### 1. Buat fungsi objektif

Kita mulai dengan membuat dictionary berisi koefisien fungsi objektif sebagai berikut:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Jalankan Optimizer
Kita selesaikan masalahnya dengan menjalankan optimizer. Karena $(x_0, ..., x_4) \in {-1, 1}^5$, kita harus mengatur `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Ambil hasilnya
Solusi dari masalah optimasi diberikan langsung dari optimizer.

In [ ]:
print(job.result())

Ini akan menampilkan dictionary dengan bentuk:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Perhatikan bahwa dictionary `solution` menampilkan vektor hasil $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Contoh 2: MaxCut

Banyak masalah graf seperti MaxCut atau Maximum Independent Set adalah masalah NP-hard dan merupakan kandidat ideal untuk menguji algoritma dan perangkat keras kuantum. Contoh ini menunjukkan cara menyelesaikan masalah MaxCut pada graf 3-regular menggunakan Quantum Optimizer.

Untuk menjalankan contoh ini, kamu harus menginstal paket `networkx` selain `qiskit-ibm-catalog`. Untuk menginstalnya, jalankan perintah berikut:

In [ ]:
# %pip install networkx numpy

### 1. Buat fungsi objektif
Mulai dengan membuat graf 3-regular acak. Untuk graf ini, kita definisikan fungsi objektif dari masalah MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Jalankan Optimizer
Selesaikan masalahnya dengan menjalankan optimizer.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Ambil hasilnya
Ambil hasilnya dan petakan bitstring solusi kembali ke node graf aslinya.

In [ ]:
print(job.result())

Solusi untuk masalah Maxcut langsung terdapat dalam sub-dictionary `solution` dari objek hasil

In [ ]:
maxcut_solution = job.result()["solution"]

## Contoh 3: Instance benchmark
Instance benchmark tersedia di GitHub: [Instance benchmark Kipu.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Instance tersebut bisa dimuat menggunakan library `pygithub`. Untuk menginstalnya, jalankan perintah berikut:

In [ ]:
# %pip install pygithub

Path untuk instance benchmark adalah:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Untuk mereproduksi performa benchmark pada instance HUBO, pilih backend `ibm_marrakesh` dan atur `direct_qubit_mapping` ke `True` pada sub-dictionary `options`.
Contoh berikut menjalankan instance Maxcut dengan 32 node.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Kasus penggunaan
Kasus penggunaan umum untuk solver Optimasi ini adalah masalah optimasi kombinatorial. Kamu bisa menyelesaikan masalah dari berbagai industri seperti keuangan, farmasi, atau logistik. Beberapa contoh berikut ini.
* Optimasi portofolio (QUBO): [publikasi ilmiah](https://doi.org/10.1103/PhysRevApplied.22.054037) dan [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Pelipatan protein (HUBO): [publikasi ilmiah](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Penjadwalan logistik (QUBO): [publikasi ilmiah](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Optimasi jaringan: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Pemisahan pasar (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Jika kamu tertarik menangani kasus penggunaan tertentu dan mengembangkan pemetaan khusus, kami siap membantu. [Hubungi kami.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## Dapatkan dukungan
Untuk dukungan, hubungi support@kipu-quantum.com.
## Langkah selanjutnya
- [Minta akses ke Quantum Optimizer dari Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Coba tutorial [Selesaikan masalah Market Split dengan Iskay Quantum Optimizer dari Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Tinjau [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Tinjau [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Tinjau [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## Informasi tambahan
Iskay, seperti nama perusahaan kami Kipu Quantum, adalah kata dari Peru. Meskipun kami adalah startup dari Jerman, kata-kata ini berasal dari tanah asal salah satu co-founder kami, di mana Quipu adalah salah satu mesin kalkulasi pertama yang dikembangkan oleh manusia 2000 tahun SM.